<img src="../Images/DSC_Logo.png" style="width: 400px;">

# Basics of Quantitative Data Analysis in Python
# - Structural Data Preprocessing

After loading and organizing data, the next step in quantitative research is often preprocessing. This means preparing the dataset so that it is consistent and ready for later analysis.

In this notebook, we focus on **structural data preprocessing** using the Iris dataset, a well-known example of tabular quantitative data. Because the original Iris dataset is already very clean, we use a deliberately messier version to explore common preprocessing steps in a more realistic way.

The focus is on:
- inspecting the dataset
- identifying data quality issues
- cleaning and standardizing values
- converting variables into appropriate formats
- checking the result after each step

While the main focus is on tabular data with `pandas`, the notebook also briefly previews how similar preprocessing ideas can be applied to arrays in Sect. 2.

In [ ]:
# Install all required libraries for this notebook:

!pip install -q -r ../requirements_B.txt   

print("The libraries have been installed.")

## 1. Preprocessing Tabular Data


We first import `pandas` and read the messy Iris CSV file. This gives us a dataframe to inspect and preprocess:

In [ ]:
import pandas as pd    # Import library (must be installed)

Iris = pd.read_csv("../Data/DataB/Iris_Messy.csv")

Iris.head()

<img src="../Images/iris.png" style="width: 780px;">

<small>*Image modified from Steve Dorand, Pixabay*</small>

## 1.1 Inspect the Dataset

Before changing anything, we **inspect** the dataset systematically. This helps us understand its structure, data types, and possible problems.

Compared to the original dataset, the messy dataset has 4 more rows:

In [ ]:
print(Iris.shape)

Compared to the original dataset, the messy dataset has an additional column (`MeasurementDate`) and the column `Petal Width (cm)` follows not the same naming convention as the other columns:

In [ ]:
print(Iris.columns)

`.dtypes` shows the data types of all columns. It is strange that the column `PetalLengthCm` has type `object`, since it should contain only numerical values.

In [ ]:
print(Iris.dtypes)

`.info()` gives a compact overview of the dataframe. Here, for example, we can quickly see that `SepalWidthCm` and `PetalLengthCm` have missing values because they have only 152 non-null entries instead of 154.

In [ ]:
Iris.info()

`.describe()` provides summary statistics, which can help reveal suspicious patterns. Here, for example, we can see missing values in `SepalWidthCm`, while `PetalLengthCm` shows an unexpected `object` type and therefore does not behave like a normal numeric column in the summary.

In addition, we see:
- There is a negative minimum in `Petal Width (cm)`, and an implausibly large maximum of 9999 in `SepalLengthCm`.
- There are 9 unique species, however, the original dataset has only 3 unique species.

In [ ]:
Iris.describe(include="all")

Let's inspect the unique labels in the `Species` column! It contains inconsistent labels such as different capitalization, spacing, and naming styles.

In [ ]:
Iris["Species"].unique()

## 1.2 Rename Columns and Labels in Columns

We want to standardize the labels in `Species` so that the same category is not represented in different ways. There are multiple ways to do this in `pandas`, for example by using **string methods** to harmonize formatting, and by using a replacement **dictionary** to map different label variants to one consistent form.

Standardize general formatting:

In [ ]:
# .str.strip() removes leading or trailing spaces
Iris["Species"] = Iris["Species"].str.strip()   

Iris["Species"].unique()

Map all variants to the final desired labels:

In [ ]:
# Create dictionary:
species_mapping = {
    "iris-versicolor": "Iris-versicolor",
    "versicolor": "Iris-versicolor",
    "setosa": "Iris-setosa",
    "Iris Setosa": "Iris-setosa",
    "iris virginica": "Iris-virginica",
    "virginica": "Iris-virginica"
}

# Use dictionary in .replace() function:
Iris["Species"] = Iris["Species"].replace(species_mapping)

Iris["Species"].unique()

One column name does not fit the naming style of the others. We use `.rename()` to rename it so that all column names are consistent. Since we only want to change a single column name, we can pass the old and new name directly as a small dictionary inside `.rename()` instead of defining a separate dictionary first.

In [ ]:
# Replace column name
Iris = Iris.rename(columns={"Petal Width (cm)": "PetalWidthCm"})

print(Iris.columns)

## 1.3 Check Missing Values

Next, we examine missing values in the dataset. Missing values are common in real-world datasets and should first be identified and understood. Depending on the context, they may later be left as missing, removed, or replaced — but the first step is always to inspect them systematically.

In `pandas`, `.isna()` and `.isnull()` can be used to detect missing entries. They do the same thing and return `True` where a value is missing and `False` otherwise. Combined with methods such as `.sum()` or row filtering, they help us count and inspect where missing values occur. 

In this messy version, missing values were introduced only in one column, but we first check the dataset systematically:

- Show the full dataframe of True/False values:

In [ ]:
Iris.isna()     # alternative: Iris.isnull()

- Count how many missing values each column contains:

In [ ]:
Iris.isna().sum()

- Show only rows that contain at least one missing value anywhere in the row:

In [ ]:
Iris[Iris.isna().any(axis=1)]

## 1.4 Clean and Convert Values

The column `PetalLengthCm` should be numeric, but some values were changed into text formats. A general workflow is to first identify problematic entries, then clean them, and finally convert the column to a numeric type.

- First, we identify values that cannot be converted directly. By trying to convert the column with `pd.to_numeric(..., errors="coerce")`, we can identify entries that are not directly numeric. Here, the entry `5.1 cm` cannot be converted because it contains text, and the missing values also appear as `NaN`.

In [ ]:
petal_length_numeric = pd.to_numeric(Iris["PetalLengthCm"], errors="coerce")

Iris[petal_length_numeric.isna()][["Id", "PetalLengthCm"]]

- We now clean the non-numeric formatting and convert the column again. The entry `5.1 cm` becomes a valid number after removing the unit, while the remaining missing values stay as `NaN`. We use string methods to remove the string `" cm"` from the value:

In [ ]:
Iris["PetalLengthCm"] = Iris["PetalLengthCm"].astype(str).str.replace(" cm", "", regex=False)

- Convert the column to numeric:

In [ ]:
Iris["PetalLengthCm"] = pd.to_numeric(Iris["PetalLengthCm"], errors="coerce")

# Check the result
print(Iris["PetalLengthCm"].dtype)
Iris.loc[Iris["PetalLengthCm"].isna(), ["Id", "PetalLengthCm"]]

## 1.5 Handle / Convert Date Formats

The `MeasurementDate` column contains dates in several different formats: `2024-05-20`, `27/08/2024`, `2024/05/31`, and `05-06-2024`. Some dates start with the year, others with the day, and both slashes and hyphens are used as separators.

In [ ]:
print(Iris["MeasurementDate"])

To handle this inconsistency, we convert the column to a proper datetime type with `to_datetime()`.
- The argument `dayfirst=True` mainly affects dates that start with day and month, especially ambiguous ones such as `05-06-2024`. 
- Dates that already clearly start with the year, such as `2024/05/31`, are still interpreted correctly, because `dayfirst=True` only matters when the parser has to decide whether the first number is the day or the month.

Because date formats can be ambiguous, it is important to inspect the converted result afterwards rather than assuming that every value was interpreted correctly.

In [ ]:
Iris["MeasurementDate"] = pd.to_datetime(
    Iris["MeasurementDate"],
    format="mixed",
    dayfirst=True,
    errors="coerce"
)

# Check the result
print(Iris["MeasurementDate"].dtype)
print(Iris["MeasurementDate"])
Iris.loc[Iris["MeasurementDate"].isna(), ["Id", "MeasurementDate"]]     # show rows where MeasurementDate is missing (because it could not be converted to datetime)

## 1.6 Implausible Numeric Values

Another useful preprocessing step is to look for values that are technically numeric but still suspicious or invalid in the context of the dataset. In this messy Iris dataset, this includes a negative petal width in column `PetalWidthCm` and the placeholder value 9999 for missing data in the column `SepalLengthCm`.

- Show rows with negative petal width:

In [ ]:
Iris.loc[Iris["PetalWidthCm"] < 0]

- Show rows where `SepalLengthCm` has the placeholder value 9999:

In [ ]:
Iris.loc[Iris["SepalLengthCm"] == 9999]

We replace these values with missing values:

In [ ]:
Iris.loc[Iris["PetalWidthCm"] < 0, "PetalWidthCm"] = pd.NA
Iris.loc[Iris["SepalLengthCm"] == 9999, "SepalLengthCm"] = pd.NA

# Check the result
print(f"Rows with missing values in PetalWidthCm:")
print(Iris.loc[Iris["PetalWidthCm"].isna(), ["Id", "PetalWidthCm"]])
print(f"\nRows with missing values in SepalLengthCm:")
print(Iris.loc[Iris["SepalLengthCm"].isna(), ["Id", "SepalLengthCm"]])

## 1.7 Duplicate Rows

Another common preprocessing step is to check whether the same observation appears more than once. Duplicate rows can result from merging files, repeated entries, or import errors. 

Duplicate rows are not automatically wrong. In some datasets, repeated rows may represent valid repeated observations. Whether duplicates should be removed depends on the meaning of the data. In this messy Iris dataset, however, exact duplicate rows are unintended and should be removed.

- Count duplicate rows:

In [ ]:
Iris.duplicated().sum()

- Show duplicate rows:

In [ ]:
Iris[Iris.duplicated(keep=False)]   # to show all occurrences of duplicated rows, not only the later repeated ones, use keep=False

Remove duplicate rows. Here, `.drop_duplicates()` keeps the first occurrence of each row and removes any exact duplicates that follow:

In [ ]:
Iris = Iris.drop_duplicates()

# Check the result
Iris.duplicated().sum()

## 1.8 Sort Data

Because the rows in the messy dataset were shuffled, we now sort the dataframe by the `Id` column to restore the familiar order of the original Iris dataset.

In [ ]:
Iris = Iris.sort_values("Id", ascending=True)

print(Iris)

If you want, you can also reset the row index afterwards:

In [ ]:
Iris = Iris.sort_values("Id").reset_index(drop=True)

print(Iris)

---
### **Exercise: Preprocessing the World Happiness Report Dataset**

In this exercise, you will apply several preprocessing steps to a messy version of the [World Happiness Report 2024 dataset](https://www.worldhappiness.report/ed/2024/) (`World-happiness-report-2024_Messy.xlsx`).

The goal is to create a cleaned and standardized dataset, sort it by `Ladder score`, and save the result as a new file.

In the messy version:
- some labels in `Regional indicator` are inconsistent,
- some missing values are written as text instead of stored as true missing values,
- and values in `Ladder score` need to be checked and cleaned before sorting.

Your tasks step-by-step:
- importing the dataset
- inspecting its structure and identifying potential issues
- applying preprocessing steps to clean and standardize the data
- sorting the cleaned dataset by `Ladder score`
- saving the cleaned version of the dataset as CSV file (see B01)

In [ ]:
# 1. Import the dataset


In [ ]:
# 2. Inspect the dataset


In [ ]:
# 3. Standardize labels in Regional indicator


In [ ]:
# 4. Replace text placeholders for missing data with true missing values


In [ ]:
# 5. Clean and convert Ladder score


In [ ]:
# 6. Sort the dataset by Ladder score


In [ ]:
# 7. Save the cleaned dataset as CSV file


---

## 2. A Brief Look at Preprocessing Data Arrays

When preprocessing is applied to numerical arrays, the general idea is very similar to preprocessing tabular data: we inspect the data, identify problematic values or patterns, and then clean or standardize them. Typical steps may include detecting missing values or placeholder codes, identifying implausible values, and replacing or masking such entries. These ideas are briefly illustrated below. The main difference is that arrays are not organized in named columns like dataframes, so preprocessing is usually done through indexing, boolean masks, and `numpy` functions.

Let's create a small numerical array. Here, the value `-999` is used as a placeholder for missing data:

In [ ]:
import numpy as np      # Import library (must be installed)

In [ ]:
data = np.array([
    [1.2, 2.5, -999],
    [2.1, 3.3, 4.1],
    [1.9, -999, 2.7]
], dtype=float)

print(data)

To replace the placeholder values with `NaN` we make use of boolean indexing:

In [ ]:
data[data == -999] = np.nan

print(data)

Check where values are missing by returning a boolean array showing which positions contain missing values:

In [ ]:
np.isnan(data)

Count how many missing values are present in the array:

In [ ]:
np.isnan(data).sum()

Show (potentially implausible) values smaller than 0. In this example, there should no longer be any negative values after replacing -999.

In [ ]:
data[data < 0]

## Key Takeaways

In this notebook, you learned that:

- preprocessing means preparing data so that it is consistent and usable for later analysis,
- a good workflow is to inspect data first, then identify problems, clean them, and check the result,
- common preprocessing steps include renaming columns, harmonizing labels, detecting missing values, converting data types, handling dates, identifying implausible values, removing duplicates, and sorting data,
- many preprocessing ideas also apply to numerical arrays.